In [89]:
import time
import cv2
import numpy as np
import easyocr
import json
import re
from skimage.filters import threshold_otsu


In [90]:
def deskew_if_needed(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)
    if lines is not None:
        angles = [np.arctan2(y2 - y1, x2 - x1) for [[x1, y1, x2, y2]] in lines]
        median_angle = np.median(angles)
        angle_deg = np.degrees(median_angle)
        if abs(angle_deg) > 1:
            (h, w) = image.shape[:2]
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle_deg, 1.0)
            image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return image

In [91]:
def fast_preprocess(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray


In [92]:
def extract_fields(results, json_format):
    extracted_data = {}
    lines = [text.strip() for (_, text, _) in results if text.strip()]
    
    dob_found = False

    for field, config in json_format.items():
        heading = config.get("Heading", "")
        direction = config.get("Direction")
        format_type = config.get("Format")
        for i, line in enumerate(lines):
            if heading.lower() in line.lower() or re.fullmatch(heading, line, re.IGNORECASE):
                try:
                    target_line = line
                    if direction == "Down" and i + 1 < len(lines):
                        target_line = lines[i + 1]
                    elif direction == "Up" and i - 1 >= 0:
                        target_line = lines[i - 1]

                    if format_type == "Pattern":
                        pattern = config.get("Pattern")
                        match = re.search(pattern, target_line)
                        if match:
                            extracted_data[field] = match.group()
                    elif format_type == "Line":
                        extracted_data[field] = target_line
                    elif format_type == "Date":
                        # Improved DOB extraction logic
                        if not dob_found:
                            # Look for 'Date of Birth' explicitly
                            if 'Date of Birth' in line or 'DOB' in line:
                                # Extract the next line or the line with the date
                                match = re.search(r"\d{2}/\d{2}/\d{4}", target_line)
                                if match:
                                    extracted_data['DOB'] = match.group()
                                    dob_found = True
                except IndexError:
                    continue
                break

    # If DOB wasn't found in the structured fields, attempt to search in general OCR results
    if not dob_found:
        for line in lines:
            if 'Date of Birth' in line or 'DOB' in line:
                match = re.search(r"\d{2}/\d{2}/\d{4}", line)
                if match:
                    extracted_data['DOB'] = match.group()
                    break

    # If DOB still isn't found, search for dates in the OCR results
    if 'DOB' not in extracted_data:
        for line in lines:
            match = re.search(r"\d{2}/\d{2}/\d{4}", line)
            if match:
                extracted_data['DOB'] = match.group()
                break

    return extracted_data

In [93]:
def process_card(image_path, json_data, card_type, reader):
    image = cv2.imread(image_path)
    image = deskew_if_needed(image)
    image = fast_preprocess(image)
    results = reader.readtext(image)
    return extract_fields(results, json_data[card_type])

In [94]:
if __name__ == "__main__":
    start = time.time()
    
    with open("format.json", "r") as f:
        json_format = json.load(f)

    card_type = "PAN_Card"
    image_path = "pancard/5.png"

    # Load EasyOCR once
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)

    extracted = process_card(image_path, json_format, card_type, reader)

    print(f"\nProcessed in {time.time() - start:.2f} seconds")
    print("Extracted Fields:")
    for k, v in extracted.items():
        print(f"{k}: {v}")


Processed in 4.46 seconds
Extracted Fields:
DOB: 16/07/1986
